# Equipo D Diferenciadores

## Experimento orientado a objetos

Este notebook incorpora primero las observaciones de `Revision_Codigo_Diferenciadores.docx`
y después los requisitos de `Integracion_Kullback_Leibler.pdf`. El experimento mantiene
GPT-2 y GPT-2-medium, el unigrama de WikiText-103 train, el fragmento fijo de test de
20 secuencias de 256 tokens, la semilla 1234 y la Gamma **forma-escala**.
WikiText-103 es el ejemplo inicial. El acceso a los corpus se configura mediante
URLs HTTP(S), formato y columna de texto, para reutilizar el mismo código en otros campos.

El estado pertenece a objetos. Las definiciones no descargan datos ni ejecutan modelos;
el flujo completo parte de `main()` en la última celda. Para una ejecución limpia,
selecciona **Reiniciar y ejecutar todas las celdas**. La inferencia completa requiere
descargar los modelos y WikiText-103; se recomienda el entorno GPU del notebook original.

### Organización de las clases

| Clase | Responsabilidad y estado propio |
|---|---|
| `FuenteCorpusURL` y `ConfiguracionCorpus` | URLs de train/test, formato, columna, codificación y caché |
| `DescargadorCorpus` | Descarga por URL, caché, integridad SHA-256 y procedencia |
| `LectorCorpusURL` | Lectura uniforme de TXT, CSV, JSON/JSONL y Parquet |
| `ConfiguracionExperimento` y `ConfiguracionTau` | Parámetros, semillas y muestras de temperatura |
| `DistribucionEmpirica` | Conteos, total de tokens y probabilidades q |
| `CorpusTexto` | Tokenizer, conteo de train y fragmento fijo de test para cualquier corpus textual |
| `MuestreadorGamma` | Validación y muestreo Gamma forma-escala |
| `ModeloConTemperatura` | Modelo, dispositivo, hook y cálculo de logits |
| `AcumuladorEvaluacion` | Estadísticas y probabilidades acumuladas por posición |
| `EvaluadorModelo` | Coordinación de inferencia y métricas por configuración |
| `CalculadoraKL` | Sumatoria, entropías, soporte y verificación empírica |
| `PresentadorResultados` | Tablas, rejillas, controles, gráficas y resúmenes |
| `ResultadoExperimento` | Tablas y textos devueltos por main |
| `ExperimentoDiferenciadores` | Preparación, ejecución ordenada y exportación |

Se usa composición: cada objeto recibe sus datos o servicios explícitamente.
Las funciones tienen anotaciones de tipos y docstrings; el experimento completo
se inicia con `main()` bajo `if __name__ == "__main__"`.

Se conservan el unigrama, los tokens frecuentes, las temperaturas, la verosimilitud,
la perplejidad, la entropía, los controles, las rejillas 3×3, los límites y el resumen.
La KL se integra en ese mismo flujo.

Los notebooks originales con sus salidas se conservan en `tmp/respaldo/`.
La ejecución exporta tablas, distribuciones promedio, figuras, validaciones e interpretación
a `resultados_diferenciadores/`. El notebook es autónomo: no necesita importar archivos locales.


### Dependencias

El entorno debe tener `torch`, `numpy`, `pandas`, `matplotlib` y `transformers`.
Para las URLs Parquet del ejemplo inicial necesita también `pyarrow`.
Si falta alguna dependencia en Colab, ejecuta en una celda aparte:

```python
%pip install torch numpy pandas matplotlib transformers pyarrow
```

La instalación se mantiene como instrucción opcional, fuera del flujo del experimento.


In [ ]:
"""Experimento del Equipo D con Gamma forma-escala y KL por sumatoria.

El notebook se genera a partir de este archivo con tools/construir_notebook.py.
Importar el módulo no descarga datos, carga modelos ni inicia experimentos.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Iterable, Iterator
from urllib.error import URLError
from urllib.parse import urlsplit
from urllib.request import Request, urlopen
import codecs
import csv
import gc
import hashlib
import itertools
import json
import math
import re
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

## 1 Configuración y muestreo Gamma

`ConfiguracionExperimento` sustituye las variables globales. `MuestreadorGamma` valida
$\alpha^*>0$, $\beta^*>0$ y usa un generador local por semilla.

$$\tau\sim\operatorname{Gamma}(\alpha^*,\beta^*)\quad\text{forma-escala},\qquad
\mathbb E[\tau]=\alpha^*\beta^*,\quad \operatorname{Var}(\tau)=\alpha^*\beta^{*2},\quad
\operatorname{CV}=1/\sqrt{\alpha^*}.$$

La rejilla conserva $\alpha^*\in\{100,250,500\}$, $\beta^*\in\{0.001,0.005,0.01\}$
y las semillas `1234 + 10*i + j`. El control se identifica por su media teórica
configurable, sin fijar un par de parámetros dentro del bucle.

### Acceso reutilizable a corpus por URL

`ConfiguracionCorpus.wikitext103()` contiene las URLs directas de los dos archivos
train y del archivo test, con revisión fija y SHA-256 del
[repositorio oficial](https://huggingface.co/datasets/Salesforce/wikitext/tree/main/wikitext-103-raw-v1).
La descarga usa HTTP(S) directamente y no depende de `datasets.load_dataset`.
Los archivos se conservan en `corpus_descargados/` y se verifica su hash antes de reutilizarlos.

Para textos de otro campo, cambia únicamente la configuración en la última celda:

```python
corpus = ConfiguracionCorpus(
    nombre="Mi corpus de otro campo",
    train=(FuenteCorpusURL("https://datos.example/train.csv", campo_texto="contenido"),),
    test=(FuenteCorpusURL("https://datos.example/test.jsonl", campo_texto="contenido"),),
    separador_test="\n",
)
configuracion = ConfiguracionExperimento(corpus=corpus)
experimento = main(configuracion)
```

Las URLs `datos.example` ilustran la configuración: reemplázalas por las de tus archivos.
Se admiten TXT, CSV, JSON (lista de registros o textos), JSONL/NDJSON y Parquet.
Si la URL no tiene extensión, indica `formato="csv"` (o el formato correspondiente).
Puedes cambiar `codificacion`, `separador_csv`, `campo_texto` y la lista ordenada de
archivos de cada split. TXT conserva los saltos de línea; CSV/JSON/Parquet leen el
campo indicado, y los valores null se tratan como texto vacío. Los registros no
textuales producen un error en lugar de convertirse silenciosamente.

Cambiar las fuentes desactiva automáticamente la comparación con las perplejidades
históricas de WikiText-103. Las verificaciones matemáticas de KL y el control con
Hugging Face se mantienen para el corpus elegido.


In [ ]:
from __future__ import annotations

@dataclass(frozen=True)
class FuenteCorpusURL:
    """Describe un archivo remoto sin acoplarlo a un proveedor, dominio o modelo."""

    url: str
    formato: str = "auto"
    campo_texto: str = "text"
    codificacion: str = "utf-8"
    separador_csv: str = ","
    sha256: str | None = None

    def __post_init__(self) -> None:
        """Valida URL HTTP(S), formato, columna, codificación y checksum opcional."""
        partes = urlsplit(self.url)
        if partes.scheme not in ("http", "https") or not partes.netloc:
            raise ValueError("El corpus debe tener una URL http:// o https:// válida")
        formato = self.formato.lower()
        if formato == "auto":
            formato = Path(partes.path).suffix.lower().lstrip(".")
        if formato == "ndjson":
            formato = "jsonl"
        if formato not in ("txt", "csv", "jsonl", "json", "parquet"):
            raise ValueError("Indica formato txt, csv, jsonl, json o parquet para esta URL")
        if not self.campo_texto or not isinstance(self.campo_texto, str):
            raise ValueError("campo_texto debe ser un nombre de columna no vacío")
        if len(self.separador_csv) != 1:
            raise ValueError("El separador CSV debe tener un carácter")
        codecs.lookup(self.codificacion)
        if self.sha256 is not None and not re.fullmatch(r"[0-9a-fA-F]{64}", self.sha256):
            raise ValueError("sha256 debe contener 64 caracteres hexadecimales")
        object.__setattr__(self, "formato", formato)
        if self.sha256 is not None:
            object.__setattr__(self, "sha256", self.sha256.lower())


@dataclass(frozen=True)
class ConfiguracionCorpus:
    """Agrupa URLs ordenadas de train/test y las opciones de acceso reutilizables."""

    train: tuple[FuenteCorpusURL, ...]
    test: tuple[FuenteCorpusURL, ...]
    nombre: str = "Corpus por URL"
    cache: Path = Path("corpus_descargados")
    separador_test: str = "\n"
    tiempo_espera: float = 30.0

    def __post_init__(self) -> None:
        """Exige fuentes explícitas para ambos splits y un timeout positivo."""
        for nombre in ("train", "test"):
            fuentes = tuple(getattr(self, nombre))
            if not fuentes or not all(isinstance(fuente, FuenteCorpusURL) for fuente in fuentes):
                raise ValueError(f"{nombre} debe contener al menos una FuenteCorpusURL")
            object.__setattr__(self, nombre, fuentes)
        if not math.isfinite(self.tiempo_espera) or self.tiempo_espera <= 0:
            raise ValueError("tiempo_espera debe ser positivo y finito")
        if not isinstance(self.separador_test, str):
            raise ValueError("separador_test debe ser texto")
        object.__setattr__(self, "cache", Path(self.cache))

    @classmethod
    def wikitext103(cls) -> ConfiguracionCorpus:
        """URLs directas oficiales, revisión fija y SHA-256 de los tres archivos."""
        base = ("https://huggingface.co/datasets/Salesforce/wikitext/resolve/"
                "b08601e04326c79dfdd32d625aee71d232d685c3/wikitext-103-raw-v1/")
        return cls(
            nombre="WikiText-103 raw v1",
            separador_test="",
            train=(
                FuenteCorpusURL(base + "train-00000-of-00002.parquet", sha256=
                    "74da360f23826045b3e6ac6375411fdb15f003030aa74f2596ed08b857cb9212"),
                FuenteCorpusURL(base + "train-00001-of-00002.parquet", sha256=
                    "ba090ac30dbf5461e8dcbdd1a1b8e6f3cf9c2c756d64f0c1220450acd514f720"),
            ),
            test=(FuenteCorpusURL(base + "test-00000-of-00001.parquet", sha256=
                  "5f1bea067869d04849c0f975a2b29c4ff47d867f484f5010ea5e861eab246d91"),),
        )

    def es_wikitext_original(self) -> bool:
        """Activa referencias históricas solo para las mismas fuentes y separación."""
        original = self.wikitext103()
        return (self.train == original.train and self.test == original.test
                and self.separador_test == original.separador_test)


@dataclass(frozen=True)
class ConfiguracionExperimento:
    """Parámetros explícitos; los valores iniciales conservan el notebook v2."""

    semilla: int = 1234
    modelos: tuple[str, ...] = ("gpt2", "gpt2-medium")
    corpus: ConfiguracionCorpus = field(default_factory=ConfiguracionCorpus.wikitext103)
    max_lineas_train: int | None = None
    lote_lineas: int = 100_000
    n_muestras: int = 20
    longitud_secuencia: int = 256
    lote_inferencia: int = 4
    alphas: tuple[float, ...] = (100, 250, 500)
    betas: tuple[float, ...] = (0.001, 0.005, 0.01)
    umbral_tau_cero: float = 1e-3
    media_control: float = 1.0
    dispositivo: str | None = None
    salida: Path = Path("resultados_diferenciadores")
    ejecutar_limites: bool = True
    alphas_limite: tuple[float, ...] = (100, 1_000, 10_000, 100_000, 1_000_000)
    betas_limite: tuple[float, ...] = (1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-8, 1e-12)
    alpha_limite: float = 100
    comprobar_referencia_historica: bool = True
    referencias_ppl: tuple[tuple[str, float], ...] = (
        ("gpt2", 49.96480), ("gpt2-medium", 38.88990),
    )

    def __post_init__(self) -> None:
        """Rechaza configuraciones vacías, no finitas o con tamaños inválidos."""
        if not isinstance(self.corpus, ConfiguracionCorpus):
            raise ValueError("corpus debe ser una ConfiguracionCorpus con URLs explícitas")
        for nombre in ("lote_lineas", "n_muestras", "longitud_secuencia", "lote_inferencia"):
            valor = getattr(self, nombre)
            if isinstance(valor, bool) or not isinstance(valor, int) or valor <= 0:
                raise ValueError(f"{nombre} debe ser un entero positivo")
        if self.n_muestras < 2:
            raise ValueError("Se necesitan al menos dos muestras para la desviación muestral")
        if self.max_lineas_train is not None and (
            isinstance(self.max_lineas_train, bool)
            or not isinstance(self.max_lineas_train, int)
            or self.max_lineas_train <= 0
        ):
            raise ValueError("max_lineas_train debe ser None o un entero positivo")
        if not self.modelos or len(set(self.modelos)) != len(self.modelos):
            raise ValueError("Debe haber modelos y sus nombres no deben repetirse")
        for nombre in ("alphas", "betas", "alphas_limite", "betas_limite"):
            valores = getattr(self, nombre)
            if not valores or any(not math.isfinite(v) or v <= 0 for v in valores):
                raise ValueError(f"{nombre} debe contener valores positivos y finitos")
            if len(set(valores)) != len(valores):
                raise ValueError(f"{nombre} no debe tener valores repetidos")
        for nombre in ("umbral_tau_cero", "media_control", "alpha_limite"):
            if not math.isfinite(getattr(self, nombre)) or getattr(self, nombre) <= 0:
                raise ValueError(f"{nombre} debe ser positivo y finito")
        object.__setattr__(self, "salida", Path(self.salida))

    def usa_fragmento_original(self) -> bool:
        """Indica si corresponde contrastar con las perplejidades históricas."""
        return (
            self.corpus.es_wikitext_original()
            and self.n_muestras == 20 and self.longitud_secuencia == 256
        )


@dataclass(frozen=True)
class ConfiguracionTau:
    """Una celda de la rejilla y sus muestras, compartidas por ambos modelos."""

    alpha: float
    beta: float
    semilla: int
    muestras: np.ndarray
    es_control: bool


class MuestreadorGamma:
    """Muestrea Gamma en parametrización forma-escala, nunca forma-tasa."""

    @staticmethod
    def muestrear(alpha: float, beta: float, n: int, semilla: int) -> np.ndarray:
        """Devuelve n valores positivos con E[tau]=alpha*beta y semilla local."""
        if not math.isfinite(alpha) or not math.isfinite(beta) or alpha <= 0 or beta <= 0:
            raise ValueError("alpha y beta deben ser positivos y finitos")
        if isinstance(n, bool) or not isinstance(n, int) or n <= 0:
            raise ValueError("n debe ser un entero positivo")
        muestras = np.random.default_rng(semilla).gamma(shape=alpha, scale=beta, size=n)
        if not np.all(np.isfinite(muestras)) or np.any(muestras <= 0):
            raise ValueError("El muestreo produjo tau inválido o subdesbordamiento a cero")
        return muestras

    def construir_rejilla(self, config: ConfiguracionExperimento) -> list[ConfiguracionTau]:
        """Conserva el orden y las semillas SEED + 10*i + j de la rejilla original."""
        return [
            ConfiguracionTau(
                alpha, beta, config.semilla + 10 * indice_alpha + indice_beta,
                self.muestrear(alpha, beta, config.n_muestras,
                               config.semilla + 10 * indice_alpha + indice_beta),
                math.isclose(alpha * beta, config.media_control, rel_tol=1e-12),
            )
            for indice_alpha, alpha in enumerate(config.alphas)
            for indice_beta, beta in enumerate(config.betas)
        ]

## 2 Distribución empírica y fragmento de test

$$q(x)=\frac{\operatorname{count}(x)}{N},\qquad\sum_xq(x)=1.$$

`DistribucionEmpirica` conserva juntos conteos, $N$ y $q$.
`CorpusTexto` mantiene la tokenización por línea de train, la unión sin separadores
del test y un BOS `<|endoftext|>` por secuencia. Ambos modelos usan el mismo vocabulario
de 50 257 tokens y predicen exactamente las 5 120 posiciones posteriores al BOS.
Se muestran también el número de tokens sin observaciones y los diez tokens más
frecuentes, con su texto e ID, como en el notebook original.
El descargador y el lector son independientes del modelo y de KL. La descarga
se realiza por bloques; Parquet se recorre por lotes y TXT/CSV/JSONL por registros.
JSON lee la lista completa. El orden de las partes y los registros se conserva.
Los corpus nuevos usan un salto de línea entre registros de test por defecto;
el ejemplo WikiText conserva la unión sin separadores de la versión original.


In [ ]:
from __future__ import annotations

class DescargadorCorpus:
    """Descarga HTTP por bloques, conserva caché verificada y registra procedencia."""

    def __init__(self, directorio: Path, tiempo_espera: float = 30.0) -> None:
        """Recibe solo parámetros de acceso, sin depender de GPT-2 ni del experimento."""
        if not math.isfinite(tiempo_espera) or tiempo_espera <= 0:
            raise ValueError("tiempo_espera debe ser positivo y finito")
        self._directorio = Path(directorio)
        self._tiempo_espera = tiempo_espera
        self._registros: dict[str, dict[str, Any]] = {}

    @property
    def registros(self) -> list[dict[str, Any]]:
        """Expone copias del manifiesto de descargas sin compartir su estado mutable."""
        return [dict(registro) for registro in self._registros.values()]

    @staticmethod
    def hash_archivo(ruta: Path) -> str:
        """Calcula SHA-256 por bloques sin cargar todo el corpus en memoria."""
        digest = hashlib.sha256()
        with ruta.open("rb") as archivo:
            for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
                digest.update(bloque)
        return digest.hexdigest()

    def _cache_valida(self, ruta: Path, manifiesto: Path,
                      fuente: FuenteCorpusURL) -> dict[str, Any] | None:
        """Comprueba tamaño y hash; un archivo parcial o alterado se descarga de nuevo."""
        try:
            registro = json.loads(manifiesto.read_text(encoding="utf-8"))
            if (registro["url"] != fuente.url or registro["formato"] != fuente.formato
                    or ruta.stat().st_size != registro["bytes"]):
                return None
            digest = self.hash_archivo(ruta)
            if digest != registro["sha256"] or (fuente.sha256 and digest != fuente.sha256):
                return None
            return registro
        except (OSError, ValueError, KeyError, TypeError):
            return None

    def descargar(self, fuente: FuenteCorpusURL) -> Path:
        """Reutiliza una copia íntegra o realiza una descarga atómica y verificable."""
        self._directorio.mkdir(parents=True, exist_ok=True)
        clave = hashlib.sha256(fuente.url.encode("utf-8")).hexdigest()
        ruta = self._directorio / f"{clave}.{fuente.formato}"
        manifiesto = self._directorio / f"{clave}.metadata.json"
        registro = self._cache_valida(ruta, manifiesto, fuente)
        if registro is not None:
            self._registros[fuente.url] = registro
            return ruta
        temporal: Path | None = None
        temporal_meta: Path | None = None
        try:
            peticion = Request(fuente.url, headers={"User-Agent": "CorpusURL/1.0"})
            with urlopen(peticion, timeout=self._tiempo_espera) as respuesta:
                largo = respuesta.headers.get("Content-Length")
                digest = hashlib.sha256()
                cantidad = 0
                with tempfile.NamedTemporaryFile(dir=self._directorio, suffix=".part", delete=False) as archivo:
                    temporal = Path(archivo.name)
                    while bloque := respuesta.read(1024 * 1024):
                        archivo.write(bloque)
                        digest.update(bloque)
                        cantidad += len(bloque)
            if largo is not None and cantidad != int(largo):
                raise ValueError("La descarga del corpus quedó incompleta")
            checksum = digest.hexdigest()
            if fuente.sha256 is not None and checksum != fuente.sha256:
                raise ValueError("El SHA-256 del corpus no coincide con el esperado")
            registro = {"url": fuente.url, "sha256": checksum, "bytes": cantidad,
                        "formato": fuente.formato, "archivo": ruta.name}
            with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=self._directorio,
                                             suffix=".part", delete=False) as archivo_meta:
                temporal_meta = Path(archivo_meta.name)
                json.dump(registro, archivo_meta, ensure_ascii=False, indent=2)
            temporal.replace(ruta)
            temporal_meta.replace(manifiesto)
            self._registros[fuente.url] = registro
            return ruta
        except (OSError, URLError) as error:
            raise RuntimeError(f"No se pudo descargar el corpus desde {fuente.url}: {error}") from error
        finally:
            if temporal is not None:
                temporal.unlink(missing_ok=True)
            if temporal_meta is not None:
                temporal_meta.unlink(missing_ok=True)


class LectorCorpusURL:
    """Convierte archivos de distintos formatos en una secuencia común de textos."""

    def __init__(self, descargador: DescargadorCorpus) -> None:
        """Recibe el componente de acceso como dependencia intercambiable."""
        self._descargador = descargador

    @property
    def registros(self) -> list[dict[str, Any]]:
        """Devuelve las URLs y hashes efectivamente utilizados en esta ejecución."""
        return self._descargador.registros

    @staticmethod
    def _extraer_texto(registro: Any, fuente: FuenteCorpusURL) -> str:
        """Lee la columna configurada; interpreta null como vacío y rechaza tipos ajenos."""
        if isinstance(registro, dict):
            if fuente.campo_texto not in registro:
                raise ValueError(f"El corpus no contiene la columna {fuente.campo_texto!r}")
            registro = registro[fuente.campo_texto]
        if registro is None:
            return ""
        if not isinstance(registro, str):
            raise ValueError("La columna seleccionada debe contener texto o null")
        return registro

    def leer(self, fuente: FuenteCorpusURL) -> Iterator[str]:
        """Itera textos de TXT, CSV, JSON, JSONL o Parquet conservando su orden."""
        ruta = self._descargador.descargar(fuente)
        if fuente.formato == "parquet":
            import pyarrow.parquet as parquet

            with parquet.ParquetFile(ruta) as archivo:
                if fuente.campo_texto not in archivo.schema_arrow.names:
                    raise ValueError(f"El corpus no contiene la columna {fuente.campo_texto!r}")
                for lote in archivo.iter_batches(batch_size=4096, columns=[fuente.campo_texto]):
                    for texto in lote.column(0).to_pylist():
                        yield self._extraer_texto(texto, fuente)
            return
        with ruta.open(encoding=fuente.codificacion, newline="") as archivo:
            if fuente.formato == "txt":
                yield from archivo
            elif fuente.formato == "csv":
                lector = csv.DictReader(archivo, delimiter=fuente.separador_csv)
                if fuente.campo_texto not in (lector.fieldnames or []):
                    raise ValueError(f"El corpus no contiene la columna {fuente.campo_texto!r}")
                for registro in lector:
                    yield self._extraer_texto(registro, fuente)
            elif fuente.formato == "jsonl":
                for numero, linea in enumerate(archivo, 1):
                    if not linea.strip():
                        continue
                    try:
                        registro = json.loads(linea)
                    except json.JSONDecodeError as error:
                        raise ValueError(f"JSONL inválido en la línea {numero}") from error
                    yield self._extraer_texto(registro, fuente)
            else:
                registros = json.load(archivo)
                if not isinstance(registros, list):
                    raise ValueError("El archivo JSON debe contener una lista de textos o registros")
                for registro in registros:
                    yield self._extraer_texto(registro, fuente)

    def leer_split(self, fuentes: tuple[FuenteCorpusURL, ...]) -> Iterator[str]:
        """Concatena las partes en el orden declarado, sin alterar sus documentos."""
        for fuente in fuentes:
            yield from self.leer(fuente)


@dataclass(frozen=True)
class DistribucionEmpirica:
    """Encapsula conteos, tamaño del corpus y q; no depende de variables globales."""

    conteos: np.ndarray
    probabilidades: np.ndarray = field(init=False)
    total_tokens: int = field(init=False)

    def __post_init__(self) -> None:
        """Valida y normaliza conteos enteros no negativos de un corpus no vacío."""
        originales = np.asarray(self.conteos)
        if (originales.ndim != 1 or originales.size == 0
                or not np.issubdtype(originales.dtype, np.integer)
                or np.any(originales < 0)):
            raise ValueError("Los conteos deben ser un vector de enteros no negativos")
        if np.any(originales > np.iinfo(np.int64).max):
            raise ValueError("Cada conteo debe caber en int64 para evitar desbordamiento")
        conteos = originales.astype(np.int64, copy=True)
        total = sum(int(valor) for valor in conteos)
        if total <= 0:
            raise ValueError("El corpus debe contener tokens")
        probabilidades = conteos.astype(np.float64) / total
        conteos.setflags(write=False)
        probabilidades.setflags(write=False)
        object.__setattr__(self, "conteos", conteos)
        object.__setattr__(self, "total_tokens", total)
        object.__setattr__(self, "probabilidades", probabilidades)


class CorpusTexto:
    """Procesa cualquier corpus textual de URLs con el tokenizer recibido."""

    def __init__(self, config: ConfiguracionExperimento, tokenizer: Any,
                 lector: LectorCorpusURL | None = None) -> None:
        """Recibe configuración, tokenizer y un lector reutilizable e independiente."""
        self.config = config
        self.tokenizer = tokenizer
        self._lector = lector if lector is not None else LectorCorpusURL(
            DescargadorCorpus(config.corpus.cache, config.corpus.tiempo_espera))

    @property
    def registros(self) -> list[dict[str, Any]]:
        """Expone la procedencia de los archivos utilizados sin compartir estado."""
        return self._lector.registros

    def contar_tokens(self, textos: Iterable[str]) -> DistribucionEmpirica:
        """Cuenta train por lotes sin cambiar la tokenización por línea original."""
        conteos = np.zeros(len(self.tokenizer), dtype=np.int64)
        iterador = iter(textos)
        if self.config.max_lineas_train is not None:
            iterador = itertools.islice(iterador, self.config.max_lineas_train)
        inicio = time.perf_counter()
        lineas = 0
        for indice_lote in itertools.count():
            lote = list(itertools.islice(iterador, self.config.lote_lineas))
            if not lote:
                break
            lineas += len(lote)
            no_vacios = [texto for texto in lote if texto.strip()]
            if not no_vacios:
                continue
            tokens = np.fromiter(itertools.chain.from_iterable(
                self.tokenizer(no_vacios)["input_ids"]), dtype=np.int64)
            conteos += np.bincount(tokens, minlength=len(conteos))
            if indice_lote % 10 == 0:
                print(f"Líneas {lineas:,}; tokens {conteos.sum():,}; "
                      f"{time.perf_counter() - inicio:.0f} s")
        return DistribucionEmpirica(conteos)

    def construir_fragmento_test(self, textos: Iterable[str]) -> torch.Tensor:
        """Devuelve (S,T+1) con BOS; usa los primeros S*T tokens y unión sin separador."""
        texto = self.config.corpus.separador_test.join(textos)
        tokens = self.tokenizer(texto)["input_ids"]
        cantidad = self.config.n_muestras * self.config.longitud_secuencia
        if len(tokens) < cantidad:
            raise ValueError("El split test no alcanza para construir el fragmento solicitado")
        bloques = torch.tensor(tokens[:cantidad], dtype=torch.long).view(
            self.config.n_muestras, self.config.longitud_secuencia)
        bos = torch.full((self.config.n_muestras, 1), self.tokenizer.eos_token_id,
                         dtype=torch.long)
        return torch.cat([bos, bloques], dim=1)

    def cargar(self) -> tuple[DistribucionEmpirica, torch.Tensor]:
        """Descarga URLs explícitas y procesa train/test sin usar load_dataset."""
        distribucion = self.contar_tokens(self._lector.leer_split(self.config.corpus.train))
        test = self.construir_fragmento_test(self._lector.leer_split(self.config.corpus.test))
        return distribucion, test

## 3 KL por sumatoria sobre el vocabulario

GPT-2 produce una distribución por contexto. Para compararla con el unigrama de train:

$$\bar p(x)=\frac1{|P|}\sum_{t\in P}p(x\mid x_{<t},w,\tau),\qquad
\operatorname{KL}(q\Vert\bar p)=\sum_{x:q(x)>0}q(x)\ln\frac{q(x)}{\bar p(x)}.$$

Este es el resultado principal, en **nats**. Se calculan también
$H(q)=-\sum q\ln q$ y $H(q,\bar p)=-\sum q\ln\bar p$ y se comprueba
$\operatorname{KL}=H(q,\bar p)-H(q)$.

La verificación empírica usa la **misma muestra de train que define q**:

$$\widehat K=\frac1N\sum_{i=1}^N\ln\frac{q(X_i)}{\bar p(X_i)}
=\frac1N\sum_{x:\operatorname{count}(x)>0}\operatorname{count}(x)
\ln\frac{\operatorname{count}(x)/N}{\bar p(x)}.$$

Agrupar por conteos evita expandir unos 118 millones de tokens y conserva exactamente
el peso $1/N$ por observación. **No se multiplica de nuevo por q.** La igualdad en la
misma muestra es algebraica; el test no valida generalización a otro corpus.

Los términos con $q=0$ se excluyen. Si $q>0$ y $\bar p=0$, KL y entropía cruzada
son infinitas y se informa la masa de q sin soporte. **No se aplica epsilon.**
Se conserva el softmax float32 del código original, la suma por lote float32 y la
acumulación float64. Los ceros por subdesbordamiento se documentan como numéricos;
para logits finitos y temperatura positiva, el softmax matemático tiene soporte completo.
La validación corrige únicamente la deriva de normalización dentro de $10^{-7}$,
dividiendo por la suma; no añade masa a los tokens de probabilidad cero.

Las pruebas 1 a 6 del PDF se ejecutan antes de descargar datos o modelos.


In [ ]:
from __future__ import annotations

class CalculadoraKL:
    """KL(q||p) en nats, sin epsilon y con manejo explícito de soporte."""

    @staticmethod
    def validar_distribucion(valores: np.ndarray) -> np.ndarray:
        """Valida y corrige deriva de normalización dentro de 1e-7, preservando ceros."""
        distribucion = np.asarray(valores, dtype=np.float64)
        if (distribucion.ndim != 1 or distribucion.size == 0
                or not np.all(np.isfinite(distribucion)) or np.any(distribucion < 0)
                or not np.isclose(distribucion.sum(), 1.0, atol=1e-7, rtol=0)):
            raise ValueError("Se requiere una distribución finita, no negativa y normalizada")
        return distribucion / distribucion.sum()

    def validar_par(self, q: np.ndarray, p: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """Verifica que ambas distribuciones estén en el mismo vocabulario."""
        q = self.validar_distribucion(q)
        p = self.validar_distribucion(p)
        if q.shape != p.shape:
            raise ValueError("q y p deben tener el mismo tamaño")
        return q, p

    def entropia(self, q: np.ndarray) -> float:
        """Calcula -sum q*ln(q), excluyendo q=0 conforme a 0*ln(0)=0."""
        q = self.validar_distribucion(q)
        soporte = q > 0
        return float(-np.sum(q[soporte] * np.log(q[soporte])))

    def entropia_cruzada(self, q: np.ndarray, p: np.ndarray) -> float:
        """Calcula -sum q*ln(p), con infinito cuando p carece de soporte para q."""
        q, p = self.validar_par(q, p)
        soporte = q > 0
        if np.any(p[soporte] == 0):
            return math.inf
        return float(-np.sum(q[soporte] * np.log(p[soporte])))

    def divergencia(self, q: np.ndarray, p: np.ndarray) -> float:
        """Resultado principal: suma una vez por token distinto, ponderando por q."""
        q, p = self.validar_par(q, p)
        soporte = q > 0
        if np.any(p[soporte] == 0):
            return math.inf
        resultado = float(np.sum(q[soporte] * (np.log(q[soporte]) - np.log(p[soporte]))))
        if resultado < -1e-10:
            raise ArithmeticError("KL negativa más allá del error de redondeo")
        return max(0.0, resultado)

    def divergencia_empirica(self, distribucion: DistribucionEmpirica,
                            p: np.ndarray) -> float:
        """Verifica (1/N)*sum_i ln[q(X_i)/p(X_i)] sobre el mismo train que define q.

        Se agrupan repeticiones por conteos para no expandir 118 millones de tokens:
        (1/N)*sum_x count(x)*ln[(count(x)/N)/p(x)]. No se multiplica otra vez por q.
        Esta igualdad es algebraica para la misma muestra; no valida generalización.
        """
        p = self.validar_distribucion(p)
        if p.shape != distribucion.conteos.shape:
            raise ValueError("Los conteos y p deben compartir vocabulario")
        observados = distribucion.conteos > 0
        if np.any(p[observados] == 0):
            return math.inf
        conteos = distribucion.conteos[observados].astype(np.float64)
        log_cociente = np.log(conteos / distribucion.total_tokens) - np.log(p[observados])
        return float(np.sum(conteos * log_cociente) / distribucion.total_tokens)

    @staticmethod
    def coinciden(primero: float, segundo: float) -> bool:
        """Compara valores finitos o dos infinitos positivos sin restar inf-inf."""
        return ((primero == math.inf and segundo == math.inf)
                or math.isclose(primero, segundo, abs_tol=1e-9, rel_tol=1e-9))

    def resumir(self, distribucion: DistribucionEmpirica, p: np.ndarray) -> dict[str, Any]:
        """Devuelve KL, entropías, masa sin soporte y las dos verificaciones."""
        q, p = self.validar_par(distribucion.probabilidades, p)
        entropia = self.entropia(q)
        cruzada = self.entropia_cruzada(q, p)
        kl = self.divergencia(q, p)
        empirica = self.divergencia_empirica(distribucion, p)
        identidad_ok = self.coinciden(kl, cruzada - entropia)
        empirica_ok = self.coinciden(kl, empirica)
        if not identidad_ok or not empirica_ok:
            raise AssertionError("Falló la identidad de entropías o la verificación empírica")
        return {
            "H(q) (nats)": entropia,
            "H(q,p̄) (nats)": cruzada,
            "KL(q‖p̄)": kl,
            "KL empírica (nats; verificación)": empirica,
            "% masa q sin soporte en p̄": 100 * float(q[(q > 0) & (p == 0)].sum()),
            "identidad H(q,p̄)-H(q) verificada": identidad_ok,
            "sumatoria-empírica verificada": empirica_ok,
        }

    def pruebas_basicas(self) -> list[dict[str, Any]]:
        """Ejecuta las pruebas 1-6 del PDF antes de cargar cualquier GPT-2."""
        ejemplo = DistribucionEmpirica(np.array([2, 2, 1, 1, 1], dtype=np.int64))
        q = ejemplo.probabilidades
        uniforme = np.full(5, 0.2)
        kl_ejemplo = self.divergencia(q, uniforme)
        sin_soporte = np.array([0.0, 0.4, 0.2, 0.2, 0.2])
        rng = np.random.default_rng(2026)
        aleatorias = [self.divergencia(rng.dirichlet(np.ones(11)),
                                      rng.dirichlet(np.ones(11))) for _ in range(100)]
        pruebas = [
            (1, "KL(q||q)=0", abs(self.divergencia(q, q)) < 1e-12),
            (2, "Corpus el perro come y el gato come", abs(kl_ejemplo - 0.0596118665558986) < 1e-12),
            (3, "H(q,p)-H(q)=KL", self.coinciden(kl_ejemplo,
                self.entropia_cruzada(q, uniforme) - self.entropia(q))),
            (4, "Falta de soporte: infinito sin NaN", self.divergencia(q, sin_soporte) == math.inf),
            (5, "Asimetría", not self.coinciden(kl_ejemplo, self.divergencia(uniforme, q))),
            (6, "No negatividad en 100 pares aleatorios", all(valor >= 0 for valor in aleatorias)),
        ]
        resultados = [{"prueba PDF": numero, "detalle": detalle, "superada": bool(ok)}
                      for numero, detalle, ok in pruebas]
        if not all(fila["superada"] for fila in resultados):
            raise AssertionError(f"Fallaron pruebas básicas de KL: {resultados}")
        return resultados

## 4 Inferencia y métricas separadas

`ModeloConTemperatura` encapsula el modelo y el hook, que siempre se retira con
`try/finally`. Se reconstruyen los logits como
$\operatorname{LN}_f(h_L)W_E^\top/\tau$ y se verifica el control contra Hugging Face.

`AcumuladorEvaluacion` reúne las estadísticas de los lotes; `EvaluadorModelo` coordina
sus resultados. Se usa `log_softmax`, y se elimina la función duplicada y sin uso
`log_p_secuencia`. El porcentaje de probabilidades efectivamente cero se distingue de
las menores que `float32.tiny`: en v2 el segundo diagnóstico tenía el nombre del primero.

La perplejidad usa las probabilidades **condicionales** de los tokens observados.
La KL compara **unigramas**, por lo que no debe confundirse con la log-verosimilitud
media ni estimarse usando los tokens de test con un peso q adicional.
La perplejidad aprovecha todo el rango de float64: se reporta infinita únicamente
si la pérdida es infinita o su exponencial desborda ese tipo numérico.


In [ ]:
from __future__ import annotations

class ModeloConTemperatura:
    """Encapsula el modelo, el dispositivo y el hook del último bloque."""

    def __init__(self, modelo: Any, dispositivo: torch.device) -> None:
        """Recibe un modelo ya cargado y lo coloca en modo de evaluación."""
        self.modelo = modelo.to(dispositivo).eval()
        self.dispositivo = dispositivo

    @classmethod
    def cargar(cls, nombre: str, dispositivo: torch.device) -> ModeloConTemperatura:
        """Descarga o recupera el modelo únicamente cuando main lo solicita."""
        from transformers import GPT2LMHeadModel

        return cls(GPT2LMHeadModel.from_pretrained(nombre), dispositivo)

    @torch.no_grad()
    def capturar_ultimo_bloque(self, tokens: torch.Tensor) -> torch.Tensor:
        """Captura h_L antes de LN_f y retira siempre el hook mediante finally."""
        captura: dict[str, torch.Tensor] = {}

        def guardar_salida(modulo: torch.nn.Module, entradas: tuple[Any, ...],
                           salida: Any) -> None:
            """Guarda la salida del bloque independientemente del envoltorio usado."""
            captura["estado"] = salida[0] if isinstance(salida, (tuple, list)) else salida

        registro = self.modelo.transformer.h[-1].register_forward_hook(guardar_salida)
        try:
            self.modelo.transformer(input_ids=tokens, use_cache=False)
        finally:
            registro.remove()
        if "estado" not in captura:
            raise RuntimeError("El último bloque no produjo una salida capturable")
        return captura["estado"]

    @torch.no_grad()
    def calcular_logits(self, estado: torch.Tensor,
                        tau: float | torch.Tensor = 1.0) -> torch.Tensor:
        """Calcula LN_f(h_L) @ W_E.T / tau con un tau positivo por secuencia."""
        temperatura = torch.as_tensor(tau, device=estado.device, dtype=estado.dtype)
        if (temperatura.ndim > 1 or (temperatura.ndim == 1 and temperatura.numel() != estado.shape[0])
                or not torch.isfinite(temperatura).all() or (temperatura <= 0).any()):
            raise ValueError("tau debe ser escalar positivo o un vector positivo por secuencia")
        if temperatura.ndim == 1:
            temperatura = temperatura.view(-1, 1, 1)
        logits = self.modelo.transformer.ln_f(estado) @ self.modelo.transformer.wte.weight.T
        logits = logits / temperatura
        if not torch.isfinite(logits).all():
            raise FloatingPointError("tau produjo logits no finitos; no se aplicó ningún epsilon")
        return logits

    @torch.no_grad()
    def verificar_logits_huggingface(self, tokens: torch.Tensor) -> float:
        """Comprueba el control tau=1 contra los logits nativos de Hugging Face."""
        referencia = self.modelo(tokens[:2], use_cache=False).logits
        manuales = self.calcular_logits(self.capturar_ultimo_bloque(tokens[:2]))
        diferencia = (referencia - manuales).abs().max().item()
        if not torch.allclose(referencia, manuales, atol=1e-3, rtol=1e-4):
            raise AssertionError(f"El control de logits tau=1 falló: {diferencia}")
        return diferencia

    @torch.no_grad()
    def perplejidad_huggingface(self, tokens: torch.Tensor, lote: int) -> float:
        """Obtiene la loss nativa por lotes, ponderada por posiciones evaluadas."""
        suma_loss = 0.0
        posiciones = 0
        for inicio in range(0, len(tokens), lote):
            bloque = tokens[inicio:inicio + lote]
            cantidad = bloque.shape[0] * (bloque.shape[1] - 1)
            suma_loss += self.modelo(bloque, labels=bloque, use_cache=False).loss.item() * cantidad
            posiciones += cantidad
        return AcumuladorEvaluacion.perplejidad_desde_perdida(suma_loss / posiciones)


@dataclass
class AcumuladorEvaluacion:
    """Acumula métricas por posición y suma probabilidades sobre el vocabulario."""

    vocabulario: int
    dispositivo: torch.device
    logp_tokens: list[torch.Tensor] = field(default_factory=list)
    entropias: list[torch.Tensor] = field(default_factory=list)
    tokens_bajo_tiny: int = 0
    tokens_con_p_cero: int = 0
    posiciones: int = 0
    suma_probabilidades: torch.Tensor = field(init=False)

    def __post_init__(self) -> None:
        """Reserva solamente un vector float64 por token del vocabulario."""
        self.suma_probabilidades = torch.zeros(self.vocabulario, dtype=torch.float64,
                                              device=self.dispositivo)

    def agregar(self, logp: torch.Tensor, objetivos: torch.Tensor) -> None:
        """Reduce un lote; distingue p=0 real de p menor al mínimo normal float32."""
        probabilidades = logp.exp()
        observadas = logp.gather(-1, objetivos.unsqueeze(-1)).squeeze(-1)
        self.logp_tokens.append(observadas)
        self.entropias.append(torch.special.entr(probabilidades).sum(-1))
        self.tokens_bajo_tiny += int((observadas < math.log(torch.finfo(torch.float32).tiny)).sum())
        self.tokens_con_p_cero += int((observadas.exp() == 0).sum())
        self.posiciones += objetivos.numel()
        # Se conserva la reducción float32 del v2 antes de acumular en float64.
        self.suma_probabilidades += probabilidades.sum((0, 1)).double()

    def distribucion_promedio(self) -> np.ndarray:
        """Calcula p_bar=(1/|P|)*sum_t p_t; corrige solo deriva de normalización."""
        if self.posiciones == 0:
            raise ValueError("No se evaluaron posiciones")
        promedio = self.suma_probabilidades / self.posiciones
        promedio = promedio / promedio.sum()
        return promedio.cpu().numpy()

    def metricas_condicionales(self) -> dict[str, float]:
        """Calcula verosimilitud, PPL, entropía y diagnósticos con las reducciones originales."""
        logp_tokens = torch.cat(self.logp_tokens)
        log_media = logp_tokens.mean().item()
        return {
            "log-verosimilitud media (token)": log_media,
            "log-verosimilitud media (secuencia)": logp_tokens.sum(-1).mean().item(),
            "perplejidad": self.perplejidad_desde_perdida(-log_media),
            "entropía media": torch.cat(self.entropias).mean().item(),
            "% log p = −∞": 100 * torch.isneginf(logp_tokens).float().mean().item(),
            "% p < tiny (float32)": 100 * self.tokens_bajo_tiny / self.posiciones,
            "% p = 0 (float32)": 100 * self.tokens_con_p_cero / self.posiciones,
        }

    @staticmethod
    def perplejidad_desde_perdida(perdida: float) -> float:
        """Calcula exp(loss) en todo el rango float64; solo desbordamiento da infinito."""
        if math.isnan(perdida):
            raise FloatingPointError("La pérdida es NaN; no se puede calcular la perplejidad")
        try:
            return math.exp(perdida)
        except OverflowError:
            return math.inf


class EvaluadorModelo:
    """Coordina inferencia, métricas condicionales, estadísticas de tau y KL."""

    def __init__(self, modelo: ModeloConTemperatura, distribucion: DistribucionEmpirica,
                 lote: int, umbral_tau_cero: float) -> None:
        """Recibe todas sus dependencias sin leer variables globales del notebook."""
        self.modelo = modelo
        self.distribucion = distribucion
        self.lote = lote
        self.umbral_tau_cero = umbral_tau_cero
        self.kl = CalculadoraKL()
        if lote <= 0 or not math.isfinite(umbral_tau_cero) or umbral_tau_cero <= 0:
            raise ValueError("El lote y el umbral de tau deben ser positivos")

    def estadisticas_tau(self, taus: np.ndarray) -> dict[str, float]:
        """Resume la temperatura sin mezclarla con la inferencia o la divergencia."""
        return {
            "τ medio": float(taus.mean()), "τ desv.": float(taus.std(ddof=1)),
            "% τ≈0": 100 * float((taus < self.umbral_tau_cero).mean()),
        }

    @torch.no_grad()
    def evaluar(self, tokens: torch.Tensor, taus: np.ndarray) -> tuple[dict[str, Any], np.ndarray]:
        """Evalúa exactamente las T posiciones después del BOS por secuencia."""
        taus = np.asarray(taus, dtype=np.float64)
        if (tokens.ndim != 2 or tokens.shape[0] < 2 or tokens.shape[1] < 2
                or taus.shape != (len(tokens),) or not np.isfinite(taus).all() or np.any(taus <= 0)):
            raise ValueError("Se requiere una matriz (S,T+1) y un tau positivo por secuencia")
        if self.modelo.modelo.config.vocab_size != len(self.distribucion.conteos):
            raise ValueError("El modelo y q deben compartir vocabulario")
        temperaturas = torch.tensor(taus, dtype=torch.float32, device=tokens.device)
        acumulador = AcumuladorEvaluacion(len(self.distribucion.conteos), tokens.device)
        for inicio in range(0, len(tokens), self.lote):
            bloque = tokens[inicio:inicio + self.lote]
            estado = self.modelo.capturar_ultimo_bloque(bloque)
            logits = self.modelo.calcular_logits(estado, temperaturas[inicio:inicio + self.lote])
            logp = F.log_softmax(logits[:, :-1].float(), dim=-1)
            acumulador.agregar(logp, bloque[:, 1:])
        promedio = acumulador.distribucion_promedio()
        return ({**self.estadisticas_tau(taus), **acumulador.metricas_condicionales(),
                 **self.kl.resumir(self.distribucion, promedio)}, promedio)

## 5 Tablas figuras e interpretación

Se reportan $H(q)$, $H(q,\bar p)$, KL por sumatoria, KL empírica de verificación,
porcentaje de masa q sin soporte, métricas originales y verificaciones para cada
configuración y cada modelo. Se guardan las figuras del efecto de tau y una figura
específica de KL contra tau medio para ambos modelos; los infinitos se identifican.

El párrafo interpretativo se redacta con los valores de la ejecución y describe
los extremos y el mínimo observado sin presuponer que la curva es monótona.
Se mantienen además los pivotes 3×3 de las métricas anteriores, la rejilla de medias
teóricas y el resumen original de extremos de temperatura, entropía y perplejidad.


In [ ]:
from __future__ import annotations

class PresentadorResultados:
    """Produce tablas, figuras e interpretación a partir de resultados ejecutados."""

    def __init__(self, salida: Path) -> None:
        """Recibe el directorio explícito de los archivos del experimento."""
        self.salida = salida

    @staticmethod
    def mostrar_tabla(tabla: pd.DataFrame) -> None:
        """Muestra una tabla en Jupyter o en consola sin modificar opciones globales."""
        try:
            from IPython.display import display
        except ImportError:
            print(tabla.to_string(index=False))
        else:
            display(tabla)

    def mostrar_rejillas(self, resultados: pd.DataFrame) -> None:
        """Conserva los pivotes por modelo e incorpora las métricas principales de KL."""
        rejilla = resultados[resultados["tipo"] == "rejilla"]
        for metrica in ("τ medio", "perplejidad", "entropía media",
                        "log-verosimilitud media (token)", "% log p = −∞",
                        "KL(q‖p̄)", "H(q,p̄) (nats)", "% masa q sin soporte en p̄"):
            for nombre, tabla in rejilla.groupby("modelo", sort=False):
                print(f"\n{nombre}: {metrica}")
                self.mostrar_tabla(tabla.pivot(index="α*", columns="β*", values=metrica).round(5))

    @staticmethod
    def resumir_metricas_originales(resultados: pd.DataFrame, vocabulario: int) -> str:
        """Conserva el resumen original de extremos de tau, entropía y perplejidad."""
        lineas = []
        rejilla = resultados[resultados["tipo"] == "rejilla"]
        for nombre, tabla in rejilla.groupby("modelo", sort=False):
            frio = tabla.loc[tabla["τ medio"].idxmin()]
            caliente = tabla.loc[tabla["τ medio"].idxmax()]
            lineas.extend([
                f"[{nombre}]",
                f"  τ más bajo ({frio['τ medio']:.3f}): entropía {frio['entropía media']:.3f}, "
                f"PPL {frio['perplejidad']:.2f}",
                f"  τ más alto ({caliente['τ medio']:.3f}): entropía {caliente['entropía media']:.3f} "
                f"(ln V = {math.log(vocabulario):.2f}), PPL {caliente['perplejidad']:.2f}",
                f"  % τ≈0 máximo: {tabla['% τ≈0'].max():.1f} % | "
                f"% log p = −∞ máximo: {tabla['% log p = −∞'].max():.2f} %",
            ])
        return "\n".join(lineas)

    @staticmethod
    def comparar_controles(resultados: pd.DataFrame) -> pd.DataFrame:
        """Contrasta cada celda de control configurable con tau=1 exacto."""
        filas = []
        for nombre, tabla in resultados.groupby("modelo", sort=False):
            referencia = tabla[tabla["tipo"] == "referencia"].iloc[0]
            for _, control in tabla[tabla["es control"]].iterrows():
                filas.append({"modelo": nombre, "α*": control["α*"], "β*": control["β*"],
                              "τ medio control": control["τ medio"],
                              "PPL control": control["perplejidad"],
                              "PPL tau=1": referencia["perplejidad"],
                              "% diferencia relativa PPL": 100 * abs(
                                  control["perplejidad"] / referencia["perplejidad"] - 1)})
        return pd.DataFrame(filas)

    def graficar(self, resultados: pd.DataFrame, vocabulario: int) -> None:
        """Guarda las figuras originales y KL contra tau para ambos modelos."""
        rejilla = resultados[resultados["tipo"] == "rejilla"]
        figura, ejes = plt.subplots(1, 3, figsize=(16, 4.5))
        metricas = ("entropía media", "perplejidad", "log-verosimilitud media (token)")
        for nombre, tabla in rejilla.groupby("modelo", sort=False):
            tabla = tabla.sort_values("τ medio")
            for eje, metrica in zip(ejes, metricas):
                eje.plot(tabla["τ medio"], tabla[metrica], "o-", label=nombre)
        ejes[0].axhline(math.log(vocabulario), ls="--", color="gray", label="ln V")
        for eje, metrica in zip(ejes, metricas):
            eje.set(xscale="log", xlabel="τ medio", title=metrica)
            eje.axvline(1, ls=":", color="gray")
            eje.grid(alpha=0.3)
            eje.legend()
        ejes[1].set_yscale("log")
        figura.tight_layout()
        figura.savefig(self.salida / "diferenciadores_tau.png", dpi=160)
        plt.show()
        plt.close(figura)

        figura, eje = plt.subplots(figsize=(10, 5))
        hay_infinitos = bool(np.isinf(rejilla["KL(q‖p̄)"].astype(float)).any())
        for indice_modelo, (nombre, tabla) in enumerate(rejilla.groupby("modelo", sort=False)):
            tabla = tabla.sort_values("τ medio")
            valores = tabla["KL(q‖p̄)"].to_numpy(dtype=float)
            finitos = np.isfinite(valores)
            linea, = eje.plot(tabla["τ medio"], np.where(finitos, valores, np.nan),
                              "o-", label=nombre)
            if not finitos.all():
                eje.scatter(tabla.loc[~finitos, "τ medio"],
                            np.full((~finitos).sum(), 0.96 - 0.08 * indice_modelo),
                            marker="^", color=linea.get_color(),
                            transform=eje.get_xaxis_transform(), clip_on=False)
            referencia = resultados[(resultados.modelo == nombre) & (resultados.tipo == "referencia")]
            eje.scatter(referencia["τ medio"], referencia["KL(q‖p̄)"], marker="x", s=65,
                        color=linea.get_color())
        eje.set(xscale="log", xlabel="τ medio (una muestra por secuencia)",
                ylabel="KL(q‖p̄) [nats]", title="Divergencia por sumatoria sobre el vocabulario")
        if hay_infinitos:
            eje.set_title("Divergencia por sumatoria sobre el vocabulario\n"
                          "Triángulos superiores: KL infinita, una fila por modelo")
        # Los marcadores usan altura del eje; matplotlib no incorpora siempre
        # sus abscisas al autoescalado. Se incluyen todas las temperaturas.
        temperaturas = resultados["τ medio"].to_numpy(dtype=float)
        eje.set_xlim(temperaturas.min() / 1.2, temperaturas.max() * 1.2)
        eje.axvline(1, ls=":", color="gray")
        eje.grid(alpha=0.3)
        eje.legend(loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)
        figura.tight_layout()
        figura.savefig(self.salida / "kl_vs_tau.png", dpi=180)
        plt.show()
        plt.close(figura)

    @staticmethod
    def interpretar(resultados: pd.DataFrame) -> str:
        """Describe los extremos y el mínimo observado sin presuponer monotonía."""
        parrafos = []
        for nombre, tabla in resultados[resultados.tipo == "rejilla"].groupby("modelo", sort=False):
            ordenados = tabla.sort_values("τ medio")
            frio, caliente = ordenados.iloc[0], ordenados.iloc[-1]
            finitos = tabla[np.isfinite(tabla["KL(q‖p̄)"].astype(float))]
            minimo = (f"El mínimo finito observado es {finitos['KL(q‖p̄)'].min():.6f} nats "
                      f"con τ medio={finitos.loc[finitos['KL(q‖p̄)'].idxmin(), 'τ medio']:.5f}. "
                      if not finitos.empty else "No hubo divergencias finitas en esta rejilla. ")
            parrafos.append(
                f"Para {nombre}, KL(q‖p̄) pasa de {frio['KL(q‖p̄)']:.6g} nats con "
                f"τ medio={frio['τ medio']:.5f} a {caliente['KL(q‖p̄)']:.6g} nats con "
                f"τ medio={caliente['τ medio']:.5f}. {minimo}"
                f"Hubo {int(np.isinf(tabla['KL(q‖p̄)'].astype(float)).sum())} configuraciones "
                "con divergencia infinita. Una temperatura pequeña concentra el softmax y puede "
                "perder soporte por subdesbordamiento numérico; una temperatura grande acerca "
                "las predicciones a la uniforme, que difiere del unigrama de train. "
                "La curva usa el promedio de las temperaturas muestreadas; configuraciones "
                "con medias similares pueden tener distintas dispersiones."
            )
        return "\n\n".join(parrafos)

## 6 Orquestación y validación del experimento

`ExperimentoDiferenciadores` ejecuta en orden las pruebas matemáticas, prepara datos,
recorre modelos y configuraciones, verifica controles y exporta resultados.
Solo conserva un modelo en memoria a la vez; los datos y las muestras de tau son comunes.

Las pruebas 7 y 8 del PDF comprueban tau=1 con KL finita y positiva y 0 % de masa
sin soporte, y la coincidencia sumatoria-empírica para todas las configuraciones.
Cada evaluación verifica además la identidad de entropías, incluidos los infinitos.

Las métricas condicionales mantienen las reducciones del v2. Se contrasta la PPL manual
de tau=1 contra la loss nativa de Hugging Face (`rtol=1e-5`) y, con el fragmento original,
contra las referencias históricas **49.96480** (gpt2) y **38.88990** (gpt2-medium),
con tolerancia relativa **0.001** para diferencias de entorno y dispositivo.
Estas referencias son datos del notebook anterior, no resultados nuevos de esta revisión.

Se conserva el estudio de límites con gpt2. Con $\alpha\to\infty$ y
$\alpha\beta=1$, el CV tiende a cero. Con $\beta\to0$ a alpha fijo, tau tiende a cero.
Los límites conjuntos $\alpha\to\infty$, $\beta\to0$ requieren especificar la trayectoria:
el producto $\alpha\beta$ no tiene por qué permanecer finito ni constante.

### Comparación con Integradores

| Cantidad | Diferenciadores forma-escala | Integradores forma-tasa |
|---|---|---|
| Densidad | $\tau^{\alpha-1}e^{-\tau/\beta}/[\Gamma(\alpha)\beta^\alpha]$ | $\beta^\alpha\tau^{\alpha-1}e^{-\beta\tau}/\Gamma(\alpha)$ |
| Media | $\alpha\beta$ | $\alpha/\beta$ |
| Varianza | $\alpha\beta^2$ | $\alpha/\beta^2$ |
| CV | $1/\sqrt\alpha$ | $1/\sqrt\alpha$ |
| NumPy | `gamma(shape=alpha, scale=beta)` | `gamma(shape=alpha, scale=1/beta)` |

En forma-tasa, $\alpha\to\infty$ y $\beta\to0$ hace que la media crezca sin límite.
La temperatura grande acerca el softmax a la uniforme, cuya entropía es $\ln(50257)$.


In [ ]:
from __future__ import annotations

@dataclass
class ResultadoExperimento:
    """Salidas de main para inspección posterior sin dependencias de celdas ocultas."""

    resultados: pd.DataFrame
    limites: pd.DataFrame
    temperaturas: pd.DataFrame
    concentracion: pd.DataFrame
    validaciones: pd.DataFrame
    controles: pd.DataFrame
    interpretacion: str
    tokens_frecuentes: pd.DataFrame = field(default_factory=pd.DataFrame)
    resumen_original: str = ""


class ExperimentoDiferenciadores:
    """Orquesta preparación, validación, modelos, rejilla, límites y exportación."""

    def __init__(self, config: ConfiguracionExperimento, tokenizer: Any | None = None,
                 lector: LectorCorpusURL | None = None) -> None:
        """Conserva configuración y servicios como atributos de instancia."""
        self.config = config
        self.dispositivo = torch.device(config.dispositivo or (
            "cuda" if torch.cuda.is_available() else "cpu"))
        self.muestreador = MuestreadorGamma()
        self.kl = CalculadoraKL()
        self.presentador = PresentadorResultados(config.salida)
        self.tokenizer = tokenizer
        self._lector_corpus = lector
        self._fuentes_corpus: list[dict[str, Any]] = []

    def preparar_datos(self) -> tuple[DistribucionEmpirica, torch.Tensor]:
        """Usa el tokenizer recibido o GPT-2 y carga los corpus mediante URLs explícitas."""
        if self.tokenizer is None:
            from transformers import GPT2TokenizerFast

            self.tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
            self.tokenizer.model_max_length = int(1e12)
        corpus = CorpusTexto(self.config, self.tokenizer, self._lector_corpus)
        datos = corpus.cargar()
        self._fuentes_corpus = corpus.registros
        return datos

    def tabla_tokens_frecuentes(self, distribucion: DistribucionEmpirica) -> pd.DataFrame:
        """Conserva los diez tokens de mayor q, con su texto si el tokenizer está disponible."""
        indices = np.argsort(-distribucion.probabilidades)[:10]
        return pd.DataFrame({
            "token_id": indices,
            "token": [repr(self.tokenizer.decode([int(indice)])) if self.tokenizer is not None
                      else f"ID {indice}" for indice in indices],
            "q(x)": distribucion.probabilidades[indices],
        })

    def tabla_temperaturas(self, rejilla: list[ConfiguracionTau]) -> pd.DataFrame:
        """Resume media, varianza, CV y el control definido por su media teórica."""
        return pd.DataFrame([{
            "α*": celda.alpha, "β*": celda.beta, "semilla": celda.semilla,
            "E[τ]=α*β*": celda.alpha * celda.beta,
            "τ medio (muestra)": float(celda.muestras.mean()),
            "Var teórica": celda.alpha * celda.beta ** 2,
            "Var muestral": float(celda.muestras.var(ddof=1)),
            "CV=1/√α*": 1 / math.sqrt(celda.alpha),
            "τ mín": float(celda.muestras.min()), "τ máx": float(celda.muestras.max()),
            "% τ≈0": 100 * float((celda.muestras < self.config.umbral_tau_cero).mean()),
            "es control": celda.es_control,
        } for celda in rejilla])

    def tabla_concentracion(self) -> pd.DataFrame:
        """Explora alpha creciente con alpha*beta=1 y las semillas originales."""
        return pd.DataFrame([{
            "α*": alpha, "β*": 1 / alpha, "E[τ]": 1.0,
            "CV teórico": 1 / math.sqrt(alpha),
            "desv. muestral": self.muestreador.muestrear(
                alpha, 1 / alpha, self.config.n_muestras, self.config.semilla).std(ddof=1),
        } for alpha in self.config.alphas_limite])

    def guardar_promedio(self, nombre: str, etiqueta: str, promedio: np.ndarray) -> str:
        """Guarda p_bar por modelo y configuración para poder auditar la sumatoria."""
        seguro = nombre.replace("/", "_").replace("\\", "_")
        ruta = self.config.salida / f"p_promedio_{seguro}_{etiqueta}.npy"
        np.save(ruta, promedio)
        return ruta.name

    def evaluar_configuraciones(self, nombre: str, evaluador: EvaluadorModelo,
                               tokens: torch.Tensor, rejilla: list[ConfiguracionTau]
                               ) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
        """Evalúa la rejilla, tau=1 exacto y, para gpt2, los límites adicionales."""
        filas: list[dict[str, Any]] = []
        limites: list[dict[str, Any]] = []
        for indice, celda in enumerate(rejilla):
            metricas, promedio = evaluador.evaluar(tokens, celda.muestras)
            filas.append({"modelo": nombre, "tipo": "rejilla", "α*": celda.alpha,
                          "β*": celda.beta, "E[τ]": celda.alpha * celda.beta,
                          "semilla": celda.semilla, "es control": celda.es_control,
                          **metricas, "nota": "CONTROL (τ≈1)" if celda.es_control else "",
                          "archivo p̄": self.guardar_promedio(nombre, f"rejilla_{indice:02}", promedio)})
        metricas, promedio = evaluador.evaluar(tokens, np.ones(len(tokens)))
        filas.append({"modelo": nombre, "tipo": "referencia", "α*": None, "β*": None,
                      "E[τ]": 1.0, "semilla": None, "es control": False, **metricas,
                      "nota": "τ = 1 exacto (GPT-2 original)",
                      "archivo p̄": self.guardar_promedio(nombre, "tau_1", promedio)})
        if self.config.ejecutar_limites and nombre == "gpt2":
            for indice, beta in enumerate(self.config.betas_limite):
                taus = self.muestreador.muestrear(self.config.alpha_limite, beta,
                                                len(tokens), self.config.semilla)
                metricas, promedio = evaluador.evaluar(tokens, taus)
                limites.append({"modelo": nombre, "tipo": "limite", "α*": self.config.alpha_limite,
                                "β*": beta, "E[τ]": self.config.alpha_limite * beta, **metricas,
                                "archivo p̄": self.guardar_promedio(nombre, f"limite_{indice:02}", promedio)})
        return filas, limites

    def validar_modelo(self, nombre: str, filas: list[dict[str, Any]],
                       ppl_hf: float) -> list[dict[str, Any]]:
        """Pruebas 7 y 8 y regresión de PPL de tau=1; falla si cambia el resultado."""
        referencia = next(fila for fila in filas if fila["tipo"] == "referencia")
        prueba_7 = (math.isfinite(referencia["KL(q‖p̄)"]) and referencia["KL(q‖p̄)"] > 0
                    and referencia["% masa q sin soporte en p̄"] == 0)
        validaciones = [{"prueba PDF": 7, "modelo": nombre,
                         "detalle": "tau=1: KL finita, positiva y masa sin soporte 0 %", "superada": prueba_7}]
        for fila in filas:
            validaciones.append({"prueba PDF": 8, "modelo": nombre,
                                 "detalle": f"Sumatoria=empírica: {fila['tipo']}, E[tau]={fila['E[τ]']}",
                                 "superada": bool(fila["sumatoria-empírica verificada"])})
        validaciones.append({"prueba PDF": "Word", "modelo": nombre,
                             "detalle": "PPL manual tau=1 coincide con loss de Hugging Face",
                             "superada": math.isclose(referencia["perplejidad"], ppl_hf, rel_tol=1e-5)})
        historicas = dict(self.config.referencias_ppl)
        if self.config.comprobar_referencia_historica and self.config.usa_fragmento_original() and nombre in historicas:
            validaciones.append({"prueba PDF": "Word", "modelo": nombre,
                                 "detalle": f"PPL histórica tau=1: {historicas[nombre]:.5f} (rtol=0.001)",
                                 "superada": math.isclose(referencia["perplejidad"], historicas[nombre], rel_tol=1e-3)})
        if not all(fila["superada"] for fila in validaciones):
            raise AssertionError(f"Falló una validación de {nombre}: {validaciones}")
        return validaciones

    def ejecutar(self) -> ResultadoExperimento:
        """Ejecuta todo en orden; conserva solo un modelo en memoria a la vez."""
        validaciones = self.kl.pruebas_basicas()
        self.config.salida.mkdir(parents=True, exist_ok=True)
        torch.manual_seed(self.config.semilla)
        print(f"Dispositivo: {self.dispositivo}; torch {torch.__version__}")
        distribucion, tokens = self.preparar_datos()
        tokens = tokens.to(self.dispositivo)
        np.save(self.config.salida / "q_corpus.npy", distribucion.probabilidades)
        np.save(self.config.salida / "conteos_corpus.npy", distribucion.conteos)
        print(f"N={distribucion.total_tokens:,}; suma q={distribucion.probabilidades.sum():.12f}; "
              f"H(q)={self.kl.entropia(distribucion.probabilidades):.6f} nats; test={tuple(tokens.shape)}")
        print(f"Tokens sin observaciones: {(distribucion.conteos == 0).sum():,} "
              f"de {len(distribucion.conteos):,}")
        rejilla = self.muestreador.construir_rejilla(self.config)
        temperaturas = self.tabla_temperaturas(rejilla)
        concentracion = self.tabla_concentracion() if self.config.ejecutar_limites else pd.DataFrame()
        filas_resultados: list[dict[str, Any]] = []
        filas_limites: list[dict[str, Any]] = []
        controles: list[dict[str, Any]] = []
        for nombre in self.config.modelos:
            inicio = time.perf_counter()
            modelo = ModeloConTemperatura.cargar(nombre, self.dispositivo)
            diferencia = modelo.verificar_logits_huggingface(tokens)
            ppl_hf = modelo.perplejidad_huggingface(tokens, self.config.lote_inferencia)
            evaluador = EvaluadorModelo(modelo, distribucion, self.config.lote_inferencia,
                                        self.config.umbral_tau_cero)
            filas, limites = self.evaluar_configuraciones(nombre, evaluador, tokens, rejilla)
            validaciones.extend(self.validar_modelo(nombre, filas, ppl_hf))
            filas_resultados.extend(filas)
            filas_limites.extend(limites)
            controles.append({"modelo": nombre, "max diferencia logits tau=1": diferencia,
                              "PPL Hugging Face": ppl_hf,
                              "PPL manual tau=1": filas[-1]["perplejidad"]})
            print(f"{nombre}: control logits={diferencia:.3g}; PPL HF={ppl_hf:.5f}; "
                  f"{time.perf_counter() - inicio:.1f} s")
            del evaluador, modelo
            gc.collect()
            if self.dispositivo.type == "cuda":
                torch.cuda.empty_cache()
        resultados = pd.DataFrame(filas_resultados)
        texto = self.presentador.interpretar(resultados)
        salida = ResultadoExperimento(resultados, pd.DataFrame(filas_limites), temperaturas,
                                      concentracion, pd.DataFrame(validaciones), pd.DataFrame(controles), texto,
                                      tokens_frecuentes=self.tabla_tokens_frecuentes(distribucion),
                                      resumen_original=self.presentador.resumir_metricas_originales(
                                          resultados, len(distribucion.conteos)))
        self.exportar(salida, distribucion)
        print("Diez tokens más frecuentes de q:")
        self.presentador.mostrar_tabla(salida.tokens_frecuentes)
        self.presentador.mostrar_tabla(salida.temperaturas.round(6))
        print("E[tau] = alpha*beta, rejilla de la asignación:")
        self.presentador.mostrar_tabla(salida.temperaturas.pivot(
            index="α*", columns="β*", values="E[τ]=α*β*"))
        self.presentador.mostrar_tabla(salida.resultados.round(6))
        self.presentador.mostrar_tabla(salida.controles)
        self.presentador.mostrar_tabla(salida.validaciones)
        self.presentador.mostrar_rejillas(salida.resultados)
        self.presentador.mostrar_tabla(self.presentador.comparar_controles(salida.resultados))
        if not salida.limites.empty:
            self.presentador.mostrar_tabla(salida.concentracion)
            self.presentador.mostrar_tabla(salida.limites.round(6))
        self.presentador.graficar(resultados, len(distribucion.conteos))
        print(salida.resumen_original)
        print(texto)
        return salida

    def exportar(self, resultado: ResultadoExperimento, distribucion: DistribucionEmpirica) -> None:
        """Exporta resultados completos y metadatos para auditar la ejecución."""
        import json
        import importlib.metadata

        for nombre, tabla in (("resultados_diferenciadores", resultado.resultados),
                              ("limites_diferenciadores", resultado.limites),
                              ("tabla_tau", resultado.temperaturas),
                              ("concentracion_tau", resultado.concentracion),
                              ("tokens_frecuentes", resultado.tokens_frecuentes),
                              ("validaciones", resultado.validaciones),
                              ("controles_tau_1", resultado.controles),
                              ("comparacion_control", self.presentador.comparar_controles(resultado.resultados))):
            if not tabla.empty:
                tabla.to_csv(self.config.salida / f"{nombre}.csv", index=False, encoding="utf-8-sig")
        (self.config.salida / "interpretacion_kl.txt").write_text(resultado.interpretacion, encoding="utf-8")
        (self.config.salida / "resumen_original.txt").write_text(resultado.resumen_original, encoding="utf-8")
        resultado.temperaturas.pivot(index="α*", columns="β*", values="E[τ]=α*β*").to_csv(
            self.config.salida / "medias_teoricas_tau.csv", encoding="utf-8-sig")
        versiones = {}
        for paquete in ("torch", "numpy", "pandas", "transformers", "pyarrow"):
            try:
                versiones[paquete] = importlib.metadata.version(paquete)
            except importlib.metadata.PackageNotFoundError:
                versiones[paquete] = "no instalado"
        metadatos = {
            "configuracion": asdict(self.config),
            "fuentes_corpus": self._fuentes_corpus,
            "versiones": versiones, "dispositivo": str(self.dispositivo),
            "tokens_train": distribucion.total_tokens, "vocabulario": len(distribucion.conteos),
            "epsilon_aplicado": False, "logaritmo": "natural", "unidad": "nat",
            "p_promedio": "softmax float32, suma por lote float32, acumulación float64 y normalización",
            "empirica": "mismos conteos de train que definen q; peso count/N sin peso q adicional",
        }
        (self.config.salida / "metadatos.json").write_text(
            json.dumps(metadatos, ensure_ascii=False, indent=2, default=str), encoding="utf-8")


def main(config: ConfiguracionExperimento | None = None) -> ResultadoExperimento:
    """Punto de entrada único: validaciones, datos, modelos, evaluación y salidas."""
    return ExperimentoDiferenciadores(config or ConfiguracionExperimento()).ejecutar()

## 7 Ejecutar todo desde main

Esta es la única celda que inicia el experimento. Para cambiar parámetros, pasa una
instancia de `ConfiguracionExperimento` a `main`, por ejemplo:

```python
configuracion = ConfiguracionExperimento(salida=Path("otra_ejecucion"))
experimento = main(configuracion)
```

`experimento` devuelve las tablas y la interpretación, sin depender de variables
ocultas de celdas anteriores. El CSV principal y los vectores `p_promedio_*.npy`
permiten auditar todos los términos de la divergencia. `validaciones.csv` registra
las pruebas realizadas; cualquier fallo detiene la ejecución con una explicación.


In [ ]:
from __future__ import annotations

if __name__ == "__main__":
    configuracion = ConfiguracionExperimento(corpus=ConfiguracionCorpus.wikitext103())
    experimento = main(configuracion)